# Random Forest Classifier — Iris Dataset

This notebook builds on `DecisionTree.ipynb` by training a **Random Forest** on the same Iris dataset, comparing its performance to a single Decision Tree, and inspecting feature importances.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load and Explore the Data

In [ ]:
data = pd.read_csv('Iris dataset.csv')
data.head()

In [ ]:
data.info()

In [ ]:
data['Species'].value_counts()

## Prepare Features and Target

In [ ]:
X = data.drop(['Species', 'Id'], axis=1)
y = data['Species']

print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

## Baseline: Single Decision Tree

For comparison, first fit a single Decision Tree (same as in `DecisionTree.ipynb`).

In [ ]:
dt = DecisionTreeClassifier(random_state=1)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print("Decision Tree Test Accuracy:", accuracy_score(y_test, y_pred_dt))

## Random Forest Classifier

Now train a Random Forest, which builds many decision trees on bootstrapped samples and averages their predictions.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train, y_train)

y_pred_train = rf.predict(X_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)

In [ ]:
print("Accuracy of Random Forest - Train:", accuracy_score(y_train, y_pred_train))
print("Accuracy of Random Forest - Test:", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=rf.classes_, yticklabels=rf.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Random Forest Confusion Matrix')
plt.show()

## Feature Importance

One advantage of Random Forests over a single Decision Tree is that they give a more stable estimate of which features matter most.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index)
plt.xlabel('Importance')
plt.title('Random Forest Feature Importances')
plt.show()

importances

## Visualize a Single Tree from the Forest

Each tree in the forest can still be inspected individually.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(15, 10))
plot_tree(rf.estimators_[0], feature_names=X.columns, class_names=rf.classes_,
          filled=True, rounded=True)
plt.title('One Tree from the Random Forest')
plt.show()

## Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 3, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=1),
                            param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("Tuned Random Forest Test Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

## Decision Tree vs. Random Forest — Accuracy Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Random Forest (Tuned)'],
    'Test Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_best)
    ]
})
comparison